# 03 · Custom collate functions and targets

`CustomCollate` does the padding and masking. **What the model learns is decided by the collate
functions**: one per feature, built with `make_collate(dtype, x=..., y=...)`:

- `x(step)`: the input value of one step (default: the step itself)
- `y(history)`: the target, from the whole training history; `history[-1]` is the event to predict,
  `history[-2]` the last one the model sees (default: `history[-1]`)

For `y`, the library ships **target strategies** for the common cases. Each one also knows how to
turn predictions back into real values (`decode`). Anything else is a plain function.

| Strategy | Target | Task | `decode` |
|---|---|---|---|
| default (`y=None`) | the next step as stored | any | — |
| `targets.NextValue(scale, shift)` | `(next - shift) / scale` | regression | `pred * scale + shift` |
| `targets.Delta(scale)` | `(next - last) / scale` | regression | `last + pred * scale` |
| `targets.LogRatio(scale)` | `log(next / last) / scale` | regression | `last * exp(pred * scale)` (stays positive) |
| `targets.Bucket(edges)` | class of the next value | classification | class id / label |
| `targets.DeltaBucket(edges)` | class of the change | classification | class id / label |
| `lambda history: ...` | anything you need | any | your own |

## 1. The contract

`CustomCollate` calls `funcs[name](values, inference)` once per batch and feature, where `values` is
that feature's history from **every sample in the batch**. It returns `(inputs, targets)`: one 1-D
tensor per sample, and one target per sample (`None` in inference mode). A collate function only
sees its own feature. Watch one being called:

In [1]:
import math
import torch
from next_sequence import CustomCollate, Feature, check_collate, make_collate, targets

def collate_spy(values, inference=False):
    print(f"called with inference={inference}, values={values}")
    return make_collate("float")(values, inference)

spy = CustomCollate([Feature("price", "regression", is_linear=True)], "price", {"price": collate_spy},
                    recent_first=False)
batch = spy([{"price": [10.0, 12.0, 15.0]}, {"price": [20.0, 18.0]}])
print("X:", batch["X"]["price"].squeeze(-1).tolist(), "| Y:", batch["Y"]["price"].tolist())

called with inference=False, values=[[10.0, 12.0, 15.0], [20.0, 18.0]]
X: [[10.0, 12.0], [20.0, 0.0]] | Y: [15.0, 18.0]


## 2. Built-in target strategies

Two price histories to try them on:

In [2]:
toy = [[10.0, 12.0, 15.0], [20.0, 18.0]]

### Next token / next value

In [3]:
print("next token:", make_collate("long")([[2, 3, 4], [5, 6]]))
next_value = targets.NextValue(scale=4.0, shift=15.0)            # mean and std from the training split
print("next value:", make_collate("float", y=next_value)(toy)[1])

next token: ([tensor([2, 3]), tensor([5])], [4, 6])
next value: [0.0, 0.75]


### Delta: the change since the last step

Scale it by the standard deviation of the changes **in the training split**: `targets.target_std`
computes it over every step. Raw changes (10, 500, 1500, ...) would give a huge MSE that drowns the
other tasks.

In [4]:
delta_std = targets.target_std(toy, targets.Delta())             # use your TRAINING histories here
delta = targets.Delta(scale=delta_std)
x, y = make_collate("float", y=delta)(toy)
print(f"delta std {delta_std:.3f} | targets {[round(v, 3) for v in y]}  <- (15-12)/std, (18-20)/std")
print("decode prediction 1.0 after a last price of 15:", delta.decode(torch.tensor([1.0]), last=[15.0]).tolist())

delta std 2.160 | targets [1.389, -0.926]  <- (15-12)/std, (18-20)/std
decode prediction 1.0 after a last price of 15: [17.160247802734375]


### LogRatio: the relative change (for values that must stay positive)

A delta adds a fixed amount whatever the level, so a predicted drop can push a price below zero.
`LogRatio` decodes with `last * exp(...)`, which is always positive.

In [5]:
log_ratio = targets.LogRatio(scale=targets.target_std(toy, targets.LogRatio()))
print("targets:", [round(v, 3) for v in make_collate("float", y=log_ratio)(toy)[1]])
print("an extreme prediction (-20) after 21.0 decodes to", log_ratio.decode(torch.tensor([-20.0]), last=[21.0]).tolist())

targets: [1.526, -0.721]
an extreme prediction (-20) after 21.0 decodes to [1.1283230781555176]


### Buckets: the next value or the change, as classes

Classes start at 0; set the feature's `n_classes` to `strategy.n_classes`. With `labels`, `decode`
returns names.

In [6]:
move = targets.DeltaBucket(edges=[-1.0, 1.0], labels=["down", "flat", "up"])
x, y = make_collate("float", y=move)(toy)
print("classes:", y, "->", [move.labels[c] for c in y], "| n_classes:", move.n_classes)
print("decode logits:", move.decode(torch.tensor([[0.1, 0.2, 3.0], [5.0, 0.0, 0.0]])))

classes: [2, 0] -> ['up', 'down'] | n_classes: 3
decode logits: ['up', 'down']


### Scaled input, raw target: `x` and `value`

Sometimes the target needs the raw value (a change in real units) while the input should be scaled.
Store both per step, e.g. `(scaled, raw)`: `x` picks the input, and every strategy's `value=` picks
the number its target is computed from.

In [7]:
steps = [[((p - 15.0) / 4.0, p) for p in history] for history in toy]     # (scaled, raw) per step
fn = make_collate("float", x=lambda s: s[0], y=targets.Delta(scale=1.0, value=lambda s: s[1]))
x, y = fn(steps)
print("inputs (scaled):", [t.tolist() for t in x], "| targets from raw prices:", y)

inputs (scaled): [[-1.25, -0.75], [1.25]] | targets from raw prices: [3.0, -2.0]


### Your own rule: a plain function

Anything the strategies don't cover is a function of the history. For example, "did the price go up
by more than 10%?" as two classes:

In [8]:
def up_10_percent(history):
    return int(history[-1] > 1.1 * history[-2])

print(make_collate("float", y=up_10_percent)(toy)[1])

[1, 0]


### Several targets from one column

Each feature has one target. To predict the price change **and** its direction, give the sample two
keys built from the same column and declare two features (section 4 does exactly this):

```python
sample = {"price": steps, "price_move": steps, ...}
features = [Feature("price", "regression", is_linear=True),
            Feature("price_move", "classification", is_linear=True, n_classes=move.n_classes)]
```

## 3. Check a collate function

`check_collate` runs both branches on a few histories and reports the first problem: different dtypes,
training inputs that don't drop exactly the last step, or a missing target. Here is a hand-written
function with two common bugs: the inference branch builds `long` inputs while training builds
`float`, and the classes start at 1 instead of 0.

In [9]:
def collate_delta_bin_handwritten(batch, inference=False):
    if inference:
        return [torch.tensor(b, dtype=torch.long) for b in batch], None   # long ...
    def logic(d):
        return 1 if d <= 1 else (2 if d <= 3 else 3)                       # classes 1..3, class 0 unused
    x = [torch.tensor(b[:-1], dtype=torch.float) for b in batch]           # ... but float here
    return x, [logic(b[-1] - b[-2]) for b in batch]

try:
    check_collate(collate_delta_bin_handwritten, [[25, 26, 27], [35, 37]])
except ValueError as e:
    print("caught:", e)

caught: training inputs are torch.float32 but inference inputs are torch.int64; both branches must build the same inputs (use make_collate)


The same idea with the library: both branches agree, classes start at 0, and `n_classes` comes from the strategy:

In [10]:
age_change = targets.DeltaBucket(edges=[1, 3])                 # <=1 -> 0, (1, 3] -> 1, >3 -> 2
collate_delta_bin = make_collate("float", y=age_change)
print(check_collate(collate_delta_bin, [[25, 26, 27], [35, 37]]), "| n_classes:", age_change.n_classes)

{'input_dtype': torch.float32, 'targets': [0, 1]} | n_classes: 3


### The most dangerous mistake is stopped automatically

A hand-written function that forgets to drop the last step in training mode gives the model the answer
as input: it would learn to copy it, score perfectly in training and be useless afterwards.
`CustomCollate` checks every training batch and stops this with an explanation:

In [11]:
def collate_forgot_to_drop_last(values, inference=False):
    xs = [torch.tensor(v, dtype=torch.float) for v in values]
    return (xs, None) if inference else (xs, [v[-1] for v in values])    # should be x[:-1]

leaky = CustomCollate([Feature("price", "regression", is_linear=True)], "price",
                      {"price": collate_forgot_to_drop_last})
try:
    leaky([{"price": [10.0, 12.0, 15.0]}])
except ValueError as e:
    print("ValueError:", e)

ValueError: funcs['price'] returned training inputs with 3 steps for sample 0, whose history has 3: the inputs still contain the last step, which is the target, so the model would learn to copy the answer. In training mode drop it (return x[:-1]) or build the function with make_collate. If your steps really hold a label for the next event, pass check_leakage=False.


If your steps store a label for the *next* event (so the full history is a legitimate input), pass
`check_leakage=False` to `CustomCollate` or `CustomCollate.from_config`.

## 4. End to end: next item + price change + direction

Three things are predicted at once from the shop data:

- `item`: the next item
- `price`: the **change** in price (`targets.Delta`, scaled on the training split)
- `price_move`: the direction of that change (`targets.DeltaBucket`)

Both price features use `(scaled log-price, raw price)` steps: the input is scaled, the targets use
real prices.

## The example data

Every cookbook uses the same small synthetic shop: each row is one purchase with an `item`,
a `price` and a `weekday`. Shoppers buy the five items in a fixed cycle
(`apple → bread → cheese → dates → eggs → apple ...`), starting at a random item, so the next
item is learnable and we can check the model gets it right.

In [12]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

from torch.utils.data import DataLoader
from next_sequence import CoreDataset, NextSequence, NextSequenceConfig, TaskType, create_bow

events = make_events()
users = (events.sort_values(["user_id", "t"]).groupby("user_id").agg(list)
               .reset_index().drop(columns="t").to_dict(orient="records"))

def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]      # target: 3rd-to-last event
val_raw = [cut(u, -1) for u in users]        # target: 2nd-to-last event
test_raw = [cut(u, None) for u in users]     # target: last event

Fit everything on the training split: the vocabulary, the input scaler and the target scale.

In [13]:
bow = create_bow(u["item"] for u in train_raw)
log_prices = [math.log1p(p) for u in train_raw for p in u["price"]]
price_mean = sum(log_prices) / len(log_prices)
price_std = (sum((v - price_mean) ** 2 for v in log_prices) / len(log_prices)) ** 0.5

train_prices = [u["price"] for u in train_raw]
delta = targets.Delta(scale=targets.target_std(train_prices, targets.Delta()), value=lambda s: s[1])
move = targets.DeltaBucket(edges=[-30, 30], labels=["down", "flat", "up"], value=lambda s: s[1])
print(f"delta scale: {delta.scale:.1f}")
print(delta)
print(move)

delta scale: 76.1
Delta(scale=76.0965221761876)
DeltaBucket(edges=[-30.0, 30.0], labels=['down', 'flat', 'up'])


In [14]:
def transform(samples):
    out = []
    for u in samples:
        steps = [((math.log1p(p) - price_mean) / price_std, p) for p in u["price"]]   # (scaled, raw)
        out.append({"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
                    "price": steps, "price_move": steps,
                    "weekday": [d + 1 for d in u["weekday"]]})
    return out

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature("price", TaskType.REGRESSION, is_linear=True),
    Feature("price_move", TaskType.CLASSIFICATION, is_linear=True, n_classes=move.n_classes),
    Feature("weekday", TaskType.PASSTHROUGH, vocab_size=8),
]
funcs = {
    "item": make_collate("long"),
    "price": make_collate("float", x=lambda s: s[0], y=delta),
    "price_move": make_collate("float", x=lambda s: s[0], y=move),
    "weekday": make_collate("long"),
}
for name in ("price", "price_move"):
    print(f"{name:10s}", check_collate(funcs[name], [s[name] for s in train[:3]]))

price      {'input_dtype': torch.float32, 'targets': [1.9447669324145482, -0.3742615192591819, 0.37820388076823225]}
price_move {'input_dtype': torch.float32, 'targets': [2, 1, 1]}


In [15]:
config = NextSequenceConfig(features=features, max_len=16, emb_dim=32, attn_heads=2)

def loader(samples, inference=False, shuffle=False, funcs=funcs):
    collate = CustomCollate.from_config(config, "item", funcs, inference=inference)
    return DataLoader(CoreDataset(samples), batch_size=32, shuffle=shuffle, collate_fn=collate)

model = NextSequence(config, epochs=20, lr=3e-3, seed=0, verbose=False)
model.fit(loader(train, shuffle=True), loader(val))
pd.DataFrame(model.history_).set_index("epoch").iloc[[0, 4, 9, 14, 19]].round(4)

,train_loss,train_item,train_price,train_price_move,val_loss,val_item,val_price,val_price_move
epoch,,,,,,,,
1,3.4128,1.8555,1.1770,0.9761,3.0106,1.6308,0.8756,0.9665
5,0.4966,0.2114,0.0538,0.2647,0.3328,0.1074,0.0360,0.2185
10,-0.0038,0.0184,0.0277,0.0912,0.0923,0.0140,0.0199,0.2005
15,-0.1258,0.0189,0.0284,0.0545,0.0695,0.0122,0.0223,0.2368
20,-0.1700,0.0135,0.0210,0.0401,0.0682,0.0118,0.0201,0.2460


### Decode the predictions with the same strategies

On the test split the model saw every step but the last, so the last known price is `price[-2]`.

In [16]:
out = model.predict(loader(test))
last_known = [u["price"][-2] for u in test_raw]
true_price = torch.tensor([u["price"][-1] for u in test_raw])

pred_price = delta.decode(out["price"], last=last_known)
pred_move = move.decode(out["price_move"])
true_move = [move.labels[move([(0, a), (0, b)])] for a, b in zip(last_known, true_price.tolist())]

baseline = (torch.tensor(last_known) - true_price).abs().mean().item()     # "price stays the same"
print(f"price MAE: {(pred_price - true_price).abs().mean():.2f}  (repeat-last-price baseline: {baseline:.2f})")
print(f"direction accuracy: {sum(p == t for p, t in zip(pred_move, true_move)) / len(true_move):.3f}")
pd.DataFrame({
    "item": [f"{u['item'][-2]} → {u['item'][-1]}" for u in test_raw[:6]],
    "last price": [round(v, 2) for v in last_known[:6]],
    "true price": true_price[:6].double().numpy().round(2),
    "predicted": pred_price[:6].double().numpy().round(2),
    "true move": true_move[:6], "predicted move": pred_move[:6],
})

price MAE: 7.88  (repeat-last-price baseline: 67.20)
direction accuracy: 0.917


,item,last price,true price,predicted,true move,predicted move
0,dates → eggs,96.94,57.72,52.98,down,down
1,apple → bread,21.14,40.51,45.26,flat,flat
2,cheese → dates,165.28,102.13,76.35,down,down
3,bread → cheese,45.87,173.41,188.68,up,up
4,cheese → dates,195.24,87.02,108.73,down,down
5,dates → eggs,97.14,61.04,71.38,down,flat


### Delta or log ratio?

Swap one strategy and compare. `LogRatio` can never decode to a negative price; whether it is more
accurate depends on the data, so compare both on your validation split.

In [17]:
log_ratio = targets.LogRatio(scale=targets.target_std(train_prices, targets.LogRatio()), value=lambda s: s[1])
funcs_ratio = {**funcs, "price": make_collate("float", x=lambda s: s[0], y=log_ratio)}

ratio_model = NextSequence(config, epochs=20, lr=3e-3, seed=0, verbose=False)
ratio_model.fit(loader(train, shuffle=True, funcs=funcs_ratio), loader(val, funcs=funcs_ratio))
ratio_price = log_ratio.decode(ratio_model.predict(loader(test, funcs=funcs_ratio))["price"], last=last_known)

pd.DataFrame({
    "target": ["Delta", "LogRatio"],
    "price MAE": [(pred_price - true_price).abs().mean().item(), (ratio_price - true_price).abs().mean().item()],
    "lowest predicted price": [pred_price.min().item(), ratio_price.min().item()],
    "negative predictions": [int((pred_price < 0).sum()), int((ratio_price < 0).sum())],
}).round(2)

,target,price MAE,lowest predicted price,negative predictions
0,Delta,7.88,18.63,0
1,LogRatio,6.34,17.71,0


## Summary

- `make_collate(dtype, x=..., y=...)` builds every collate function; check custom ones with `check_collate`.
- Use a **target strategy** for `y` when one fits, and keep it: its `decode` turns predictions back.
- Fit `scale` with `targets.target_std(train_histories, Strategy())` on the **training split**.
- Classes start at 0; take `n_classes` from the bucket strategy.
- A change-based target needs at least two steps of history.
- For values that must stay positive, prefer `LogRatio` (or the `positive_regression` task, see 07).